In [46]:
import pandas as pd

In [47]:
df=pd.read_csv("AI_Resume_Screening.csv")

In [48]:
print(df.head())

   Resume_ID              Name                                        Skills  \
0          1        Ashley Ali                      TensorFlow, NLP, Pytorch   
1          2      Wesley Roman  Deep Learning, Machine Learning, Python, SQL   
2          3     Corey Sanchez         Ethical Hacking, Cybersecurity, Linux   
3          4  Elizabeth Carney                   Python, Pytorch, TensorFlow   
4          5        Julie Hill                              SQL, React, Java   

   Experience (Years) Education                Certifications  \
0                  10      B.Sc                           NaN   
1                  10       MBA                     Google ML   
2                   1       MBA  Deep Learning Specialization   
3                   7    B.Tech                 AWS Certified   
4                   4       PhD                           NaN   

                Job Role Recruiter Decision  Salary Expectation ($)  \
0          AI Researcher               Hire              

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Resume_ID               1000 non-null   int64 
 1   Name                    1000 non-null   object
 2   Skills                  1000 non-null   object
 3   Experience (Years)      1000 non-null   int64 
 4   Education               1000 non-null   object
 5   Certifications          726 non-null    object
 6   Job Role                1000 non-null   object
 7   Recruiter Decision      1000 non-null   object
 8   Salary Expectation ($)  1000 non-null   int64 
 9   Projects Count          1000 non-null   int64 
 10  AI Score (0-100)        1000 non-null   int64 
dtypes: int64(5), object(6)
memory usage: 86.1+ KB


In [50]:
df1=pd.read_csv("UpdatedResumeDataSet.csv")

In [51]:
df1.head()

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."


In [52]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  962 non-null    object
 1   Resume    962 non-null    object
dtypes: object(2)
memory usage: 15.2+ KB


In [53]:
df.isnull().sum()

Resume_ID                   0
Name                        0
Skills                      0
Experience (Years)          0
Education                   0
Certifications            274
Job Role                    0
Recruiter Decision          0
Salary Expectation ($)      0
Projects Count              0
AI Score (0-100)            0
dtype: int64

In [54]:
df1.isnull().sum()

Category    0
Resume      0
dtype: int64

In [55]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d{10,}", " ", text)
    text = re.sub(r"[^a-zA-Z+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [56]:
df1["clean_resume"] = df1["Resume"].apply(clean_text)

df1 = df1.drop_duplicates(subset="clean_resume").reset_index(drop=True)

print(df1.shape)

(166, 3)


In [57]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer # changes word into number
from sklearn.linear_model import LogisticRegression # learns category pattern
from sklearn.pipeline import Pipeline # puts steps togetter
from sklearn.metrics import classification_report # prints reportcard for your model

In [58]:
x=df1["Resume"].apply(clean_text)
y=df1["Category"]

In [59]:
x_train,x_test,y_train,y_test=train_test_split(
    x,y, test_size=0.2,random_state=42,stratify=y
)

In [60]:
model= Pipeline([
    ("tfidf",TfidfVectorizer(stop_words="english",max_features=5000)), #pipelines creates the learning path tfidf converts the words into number and the linear regression learns the pattern
    ("classifier",LogisticRegression(max_iter=2000))
])

In [61]:
model.fit(x_train,y_train)
predictions=model.predict(x_test)
print(classification_report(y_test,predictions))

                           precision    recall  f1-score   support

                 Advocate       0.67      1.00      0.80         2
                     Arts       0.00      0.00      0.00         1
       Automation Testing       0.33      1.00      0.50         1
               Blockchain       0.00      0.00      0.00         1
         Business Analyst       0.00      0.00      0.00         1
           Civil Engineer       1.00      1.00      1.00         1
             Data Science       1.00      1.00      1.00         2
                 Database       0.67      1.00      0.80         2
          DevOps Engineer       0.00      0.00      0.00         1
         DotNet Developer       1.00      0.50      0.67         2
            ETL Developer       1.00      1.00      1.00         1
   Electrical Engineering       1.00      1.00      1.00         1
                       HR       0.67      1.00      0.80         2
                   Hadoop       1.00      1.00      1.00     

C:\Users\Aditya Pratap Singh\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\Aditya Pratap Singh\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
C:\Users\Aditya Pratap Singh\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\Aditya Pr

In [62]:
import joblib
joblib.dump(model, "category_model.pkl")

['category_model.pkl']

In [63]:
SKILLS = [ # skill extractor
    "python", "java", "sql", "machine learning", "deep learning",
    "tensorflow", "pytorch", "pandas", "numpy", "excel", "tableau",
    "power bi", "react", "node.js", "docker", "aws", "html", "css"
]

def extract_skills(text):
    text = text.lower()
    return [skill for skill in SKILLS if skill in text]

In [64]:
extract_skills("I know Python, SQL, Docker and AWS")

['python', 'sql', 'docker', 'aws']

In [65]:

def match_resume_to_job(resume_text, job_description):
    resume_skills = set(extract_skills(resume_text))
    job_skills = set(extract_skills(job_description))

    matched = resume_skills.intersection(job_skills)
    missing = job_skills - resume_skills

    score = 0 if not job_skills else (len(matched) / len(job_skills)) * 100

    return {
        "score": round(score, 2),
        "matched_skills": list(matched),
        "missing_skills": list(missing)
    }

In [66]:
job = "Need Python, SQL, Machine Learning and Tableau"
resume = "I have Python, SQL and Machine Learning experience and tableau"

print(match_resume_to_job(resume, job))

{'score': 100.0, 'matched_skills': ['python', 'sql', 'machine learning', 'tableau'], 'missing_skills': []}


In [67]:
df.sample(5)

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
950,951,Rita Russell,"Deep Learning, Machine Learning",7,M.Tech,AWS Certified,Data Scientist,Hire,87562,9,100
163,164,Steven Johnson,"TensorFlow, NLP, Python",1,M.Tech,NaN,AI Researcher,Reject,76442,2,35
15,16,Courtney Cook,"Cybersecurity, Networking",1,PhD,Google ML,Cybersecurity Analyst,Reject,51388,6,60
717,718,Melissa Patterson,"Networking, Ethical Hacking, Linux, Cybersecurity",4,PhD,NaN,Cybersecurity Analyst,Hire,52790,2,70
887,888,Debra Williams,"Ethical Hacking, Cybersecurity, Linux",1,B.Sc,Deep Learning Specialization,Cybersecurity Analyst,Reject,117772,2,45


In [68]:
df1.sample(5)

,Category,Resume,clean_resume
158,Blockchain,"SKILLS Bitcoin, Ethereum Solidity Hyperledger,...",skills bitcoin ethereum solidity hyperledger b...
6,Data Science,Skills â¢ Python â¢ Tableau â¢ Data Visuali...,skills python tableau data visualization r stu...
0,Data Science,Skills * Programming Languages: Python (pandas...,skills programming languages python pandas num...
88,Automation Testing,SOCIAL SKILLS: â¢ Ability to establish trust ...,social skills ability to establish trust and w...
139,Hadoop,Technical Skill Set: Programming Languages Apa...,technical skill set programming languages apac...


In [69]:
df["Recruiter Decision"].value_counts()

Recruiter Decision
Hire      812
Reject    188
Name: count, dtype: int64

In [70]:
df["Certifications"] = df["Certifications"].fillna("None")

In [71]:
features = [
    "Skills",
    "Experience (Years)",
    "Education",
    "Certifications",
    "Job Role",
    "Projects Count"
]

target = "Recruiter Decision"

In [73]:
x=df[features]
y=df[target]

x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
from sklearn.compose import ColumnTransformer #combines different cleaning method
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder # convert category words into number
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # fills empty values
from sklearn.linear_model import LogisticRegression # ml algo model that lears to predict

In [75]:
text_column = "Skills"

category_columns = [
    "Education",
    "Certifications",
    "Job Role"
]

number_columns = [
    "Experience (Years)",
    "Projects Count"
]

In [78]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "skills_text",
            TfidfVectorizer(stop_words="english"),
            text_column
        ),
        (
            "categories",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            category_columns
        ),
        (
            "numbers",
            SimpleImputer(strategy="median"),
            number_columns
        )
    ]
)

In [79]:
screening_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

screening_model.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('skills_text', ...), ('categories', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [80]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = screening_model.predict(x_test)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

        Hire       0.99      0.96      0.98       162
      Reject       0.86      0.97      0.91        38

    accuracy                           0.96       200
   macro avg       0.93      0.97      0.95       200
weighted avg       0.97      0.96      0.97       200

[[156   6]
 [  1  37]]


In [81]:
import joblib

joblib.dump(screening_model, "screening_model.pkl")

['screening_model.pkl']

In [82]:
test_candidate = pd.DataFrame([{
    "Skills": "Python, SQL, Pandas, Machine Learning, TensorFlow",
    "Experience (Years)": 3,
    "Education": "B.Tech",
    "Certifications": "Google ML",
    "Job Role": "Data Scientist",
    "Projects Count": 4
}])

print(screening_model.predict(test_candidate))
print(screening_model.predict_proba(test_candidate))

['Hire']
[[0.94592523 0.05407477]]
